<div class="alert alert-success py-2 px-3 mb-2"
     style="font-family: 'Montserrat', sans-serif; font-size: 13px;">

<style>
.alert code {
    background: transparent !important;
    color: inherit !important;
    padding: 0 !important;
    font-family: 'Inter', sans-serif !important;
    font-size: 13px !important;
}
</style>

<strong>Lab 03 – Datasets Operations</strong>

<strong>Notes:</strong> <code>notes/03-datasets-operations.md</code> ·
<strong>Result:</strong> <code>results/03-datasets-operations.md</code>

<strong>Goal:</strong> Load, inspect, transform, save, and push a dataset.

<strong>Needs:</strong> <code>HF_WRITE_TOKEN</code> in the repository-root <code>.env</code> file.

<strong>Install:</strong> <code>pip install datasets</code>

<strong>Warning:</strong> The Hub repository and local files created by this lab are deleted in the cleanup cell.

</div>

In [8]:
! uv pip uninstall numpy scipy pandas
! uv pip install "numpy>=2.0.2" "scipy>=1.14" "pandas>=3.1.1"

Using Python 3.12.11 environment at: /home/zeus/miniconda3/envs/cloudspace
Uninstalled 3 packages in 433ms
 - numpy==1.26.4
 - pandas==3.1.1
 - scipy==1.13.1
Using Python 3.12.11 environment at: /home/zeus/miniconda3/envs/cloudspace
Resolved 5 packages in 15ms                                          
Installed 3 packages in 180ms                               
 + numpy==2.5.5
 + pandas==3.1.1
 + scipy==1.18.1


In [9]:
import numpy as np
import pandas as pd
import scipy

print("numpy:", np.__version__)
print("pandas:", pd.__version__)
print("scipy:", scipy.__version__)

numpy: 2.5.5
pandas: 3.1.1
scipy: 1.18.1


In [10]:
# ── Lab 03 - Datasets Operations ──
# Notes : notes/03-datasets-operations.md
# Result: results/03-datasets-operations.md
# Goal  : load, inspect, transform, save and push a dataset
# Needs : HF_WRITE_TOKEN in the repo-root .env
# Install: pip install datasets
# Note  : the Hub repo and local files are deleted in the cleanup cell

# ── imports ──
import os, shutil
from pathlib import Path
from dotenv import load_dotenv
import datasets
from datasets import Dataset, DatasetDict, load_dataset, load_from_disk
from huggingface_hub import HfApi

# ── load the write token from .env ──
load_dotenv()
WRITE_TOKEN = os.environ["HF_WRITE_TOKEN"]
os.environ.pop("HF_TOKEN", None)

# ── user name and the repo we will create later ──
api = HfApi(token=WRITE_TOKEN)
USER = api.whoami()["name"]
DATASET_REPO = f"{USER}/datasets-lab-tiny"

print("datasets version:", datasets.__version__)
print("dataset repo    :", DATASET_REPO)

datasets version: 5.2.0
dataset repo    : akshayredekar74/datasets-lab-tiny


In [11]:
# ── 1. build a dataset from a dictionary ──
# Each key is a column, each list holds that column's values

data = {
    "text": [
        "great movie",
        "terrible acting and a boring plot",
        "loved every minute",
        "not worth the money",
        "a true masterpiece of cinema",
        "I fell asleep",
        "funny and warm",
        "the ending ruined it",
        "beautiful music",
        "too long and too loud",
        "would watch again",
        "a complete mess",
    ],
    "label": [1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0],   # 1 = good, 0 = bad
}

ds = Dataset.from_dict(data)
print(ds)

Dataset({
    features: ['text', 'label'],
    num_rows: 12
})


In [12]:
# ── 2. inspect a dataset ──

print("rows    :", ds.num_rows)
print("columns :", ds.column_names)
print("features:", ds.features)

# ── one row, several rows, one column ──
print("row 0   :", ds[0])
print("rows 0-1:", ds[:2])
print("texts   :", ds["text"][:3])

rows    : 12
columns : ['text', 'label']
features: {'text': Value('string'), 'label': Value('int64')}
row 0   : {'text': 'great movie', 'label': 1}
rows 0-1: {'text': ['great movie', 'terrible acting and a boring plot'], 'label': [1, 0]}
texts   : ['great movie', 'terrible acting and a boring plot', 'loved every minute']


In [13]:
# ── 3. load a public dataset ──
# split="train[:200]" means: only the first 200 rows

movies = load_dataset(
    "cornell-movie-review-data/rotten_tomatoes",
    split="train[:200]",
)

print(movies)
print("first row:", movies[0])

train.parquet: reconstructing file:   0%|          |  0.00B /  699kB            

train.parquet: downloading bytes:           |  0.00B            

validation.parquet: reconstructing file:   0%|          |  0.00B / 90.0kB            

validation.parquet: downloading bytes:           |  0.00B            

test.parquet: reconstructing file:   0%|          |  0.00B / 92.2kB            

test.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/8530 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1066 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1066 [00:00<?, ? examples/s]

Dataset({
    features: ['text', 'label'],
    num_rows: 200
})
first row: {'text': 'the rock is destined to be the 21st century\'s new " conan " and that he\'s going to make a splash even greater than arnold schwarzenegger , jean-claud van damme or steven segal .', 'label': 1}


In [14]:
# ── 4. map: add a new column ──
# map runs a function on every row

def add_word_count(row):
    return {"n_words": len(row["text"].split())}

ds = ds.map(add_word_count)
print(ds.column_names)
print(ds[0])

Map:   0%|          | 0/12 [00:00<?, ? examples/s]

['text', 'label', 'n_words']
{'text': 'great movie', 'label': 1, 'n_words': 2}


In [15]:
# ── 5. filter: keep only some rows ──
# The function returns True for rows we want to keep

long_reviews = ds.filter(lambda row: row["n_words"] >= 4)
print("before:", ds.num_rows, "rows")
print("after :", long_reviews.num_rows, "rows")
print(long_reviews["text"])

Filter:   0%|          | 0/12 [00:00<?, ? examples/s]

before: 12 rows
after : 5 rows
Column(['terrible acting and a boring plot', 'not worth the money', 'a true masterpiece of cinema', 'the ending ruined it', 'too long and too loud'])


In [16]:
# ── 6. select, shuffle, sort ──

# ── pick rows by position ──
print("first 3:", ds.select(range(3))["text"])

# ── shuffle (seed makes it repeatable) ──
print("shuffled:", ds.shuffle(seed=42)["text"][:3])

# ── sort by a column ──
print("shortest:", ds.sort("n_words")["text"][:3])

first 3: Column(['great movie', 'terrible acting and a boring plot', 'loved every minute'])
shuffled: ['great movie', 'the ending ruined it', 'funny and warm']
shortest: ['great movie', 'beautiful music', 'loved every minute']


/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/pyarrow/compute.py:231: FutureWarning: Specifying null_placement in SortOptions is deprecated as of 25.0.0. Specify null_placement per sort_key instead.
  return options_class(*args, **kwargs)


In [17]:
# ── 7. rename and remove columns ──

renamed = ds.rename_column("label", "sentiment")
print("renamed:", renamed.column_names)

smaller = renamed.remove_columns(["n_words"])
print("smaller:", smaller.column_names)

renamed: ['text', 'sentiment', 'n_words']
smaller: ['text', 'sentiment']


In [18]:
# ── 8. train / test split ──
# Gives a DatasetDict with two parts

split = ds.train_test_split(test_size=0.25, seed=42)
print(split)
print("train rows:", split["train"].num_rows)
print("test rows :", split["test"].num_rows)

DatasetDict({
    train: Dataset({
        features: ['text', 'label', 'n_words'],
        num_rows: 9
    })
    test: Dataset({
        features: ['text', 'label', 'n_words'],
        num_rows: 3
    })
})
train rows: 9
test rows : 3


In [19]:
# ── 9. save to disk ──
# save_to_disk keeps everything (columns, types, splits)

split.save_to_disk("tiny_dataset")
reloaded = load_from_disk("tiny_dataset")
print("reloaded:", reloaded)

Saving the dataset (0/1 shards):   0%|          | 0/9 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/3 [00:00<?, ? examples/s]

reloaded: DatasetDict({
    train: Dataset({
        features: ['text', 'label', 'n_words'],
        num_rows: 9
    })
    test: Dataset({
        features: ['text', 'label', 'n_words'],
        num_rows: 3
    })
})


In [23]:
# ── 10. export to files ──
from pathlib import Path

results_dir = Path("results")
results_dir.mkdir(parents=True, exist_ok=True)

ds.to_csv(results_dir / "tiny.csv")
ds.to_parquet(results_dir / "tiny.parquet")


# ── load the CSV back ──
from_csv = load_dataset(
    "csv",
    data_files=str(results_dir / "tiny.csv"),
    split="train",
)

# ── a pandas DataFrame, if you like tables ──
ds.to_pandas().head()

Creating CSV from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Generating train split: 0 examples [00:00, ? examples/s]

,text,label,n_words
0,great movie,1,2
1,terrible acting and a boring plot,0,6
2,loved every minute,1,3
3,not worth the money,0,4
4,a true masterpiece of cinema,1,5


In [26]:
# ── 11. streaming ──
# Reads rows one by one, without downloading everything first.
# Useful for huge datasets.

stream = load_dataset("cornell-movie-review-data/rotten_tomatoes", split="train", streaming=True)

for row in stream.take(3):
    print(row["label"], "|", row["text"][:60])

1 | the rock is destined to be the 21st century's new " conan " 
1 | the gorgeously elaborate continuation of " the lord of the r
1 | effective but too-tepid biopic


In [24]:
# ── 12. push to the Hub ──
# Uploads the split as a private dataset repo

split.push_to_hub(DATASET_REPO, private=True, token=WRITE_TOKEN)
print("pushed: https://huggingface.co/datasets/" + DATASET_REPO)
print("files  :", api.list_repo_files(DATASET_REPO, repo_type="dataset"))

Setting num_proc from 1 back to 1 for the train split to disable multiprocessing as it only contains one shard.


Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Validating:           |   0%            

... (3 outputs left out)

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Validating:           |   0%            

pushed: https://huggingface.co/datasets/akshayredekar74/datasets-lab-tiny
files  : ['.gitattributes', 'README.md', 'data/test-00000-of-00001.parquet', 'data/train-00000-of-00001.parquet']


In [25]:
# ── 13. load it back from the Hub ──
# Private repo, so we pass the token

from_hub = load_dataset(DATASET_REPO, token=WRITE_TOKEN)
print(from_hub)
print("first train row:", from_hub["train"][0])

README.md:   0%|          | 0.00/420 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 1.95kB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 1.78kB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/9 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/3 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label', 'n_words'],
        num_rows: 9
    })
    test: Dataset({
        features: ['text', 'label', 'n_words'],
        num_rows: 3
    })
})
first train row: {'text': 'too long and too loud', 'label': 0, 'n_words': 5}
